# 1 - Spark Concepts and Definitions

## What is Apache Spark

Apache Spark is a distributed data processing engine designed to process large-scale data efficiently across clusters of machines.
Spark enables developers to perform:
- Batch processing
- Streaming processing
- Machine learning
- Graph processing
- SQL analytics

Spark is widely used in modern Data Engineering architectures such as Data Lakes and Lakehouses.

Key characteristics of Apache Spark:
- Distributed computing
- In-memory processing
- Fault tolerance
- Scalability

Spark applications run on clusters and distribute computations across multiple nodes.

## Spark Core Components

A Spark application is composed of several components working together.

The most important ones are:
- Driver
- Executors
- Tasks
- Jobs
- Stages
- Partitions

Understanding these components is critical for performance optimization and troubleshooting.

### Driver

The Driver is the process that runs the main Spark application.

Responsibilities of the Driver:
- Creates the SparkSession
- Translates user code into execution plans
- Coordinates the execution of tasks
- Communicates with the cluster manager

Example:

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("spark-study").getOrCreate()


In this example, the SparkSession is created in the Driver program.

### Executors

Executors are worker processes responsible for executing tasks.

Each executor:
- Runs on worker nodes
- Executes tasks assigned by the Driver
- Stores data in memory or disk
- Reports results back to the Driver

Example cluster structure:

```
Driver
   |
   |---- Executor 1
   |---- Executor 2
   |---- Executor 3
```

Each executor can process multiple tasks in parallel.


### Jobs

A Job is triggered whenever an Action is executed.

Examples of Spark Actions
- count()
- collect()
- show()
- write()

Example:

In [0]:
df = spark.createDataFrame([
    (1, "Alice", 23),
    (2, "Bob", 34)
], ["id", "name", "age"])

df.count()

When this action (- df.count() - ) runs, Spark creates a job.

### Stages

Each job is divided into stages.

Stages represent sets of tasks that can be executed without requiring data reshuffling.

Stages are separated by shuffle boundaries.

Example operations that create stages:
- groupBy
- join
- distinct
- repartition

### Tasks

A Task is the smallest unit of execution in Spark.

Each task processes one partition of data.

Example:

If a DataFrame has:

```
8 partitions
```

Spark will create:
```
8 tasks
```
Tasks run in parallel across executors.

### Partitions

Partitions are how Spark splits data for distributed processing.

Each partition:
- Is processed independently
- Can run in parallel
- Is processed by a single task

Example:

In [0]:
df = spark.createDataFrame([
    (1, "Alice", 23),
    (2, "Bob", 34)
], ["id", "name", "age"])

# You will get an error when running this command in the free edition, as parallelization is only available in the paid version.
# df.rdd.getNumPartitions()

This returns the number of partitions of the DataFrame.

### Lazy Evaluation (Narrow transformations)

Spark uses Lazy Evaluation, meaning transformations are not executed immediately.
Instead, Spark builds a logical execution plan.
Execution only happens when an action is triggered.

<b>Transformations</b>

Transformations create a new DataFrame but do not execute immediately.

Examples:
- select()
- filter()
- withColumn()
- groupBy()
- join()

Example:

In [0]:
df = spark.createDataFrame([
    (1, "itemA", 120),
    (2, "itemB", 80),
    (3, "itemC", 150)
], ["id", "name", "price"])

df_filtered = df.filter("price > 100")
# No computation happens yet.

No computation happens yet.

### Actions

Actions trigger the execution of the DAG.

Examples:
- show()
- count()
- collect()
- write()

Example:

In [0]:
df_filtered.count()
# Now Spark executes the computation.

Now Spark executes the computation.

### Directed Acyclic Graph (DAG)

Spark builds a DAG (Directed Acyclic Graph) representing all transformations required to compute a result.

Example transformation chain:
```
read → filter → select → groupBy → agg
```
Spark converts this into an optimized execution plan.

This optimization is performed by the <b>Catalyst Optimizer</b>.

### Catalyst Optimizer

The Catalyst Optimizer is the query optimization framework used by Spark SQL.

Catalyst performs multiple optimizations before executing queries.

Execution phases:
- Logical Plan
- Optimized Logical Plan
- Physical Plan
- Execution

Example to visualize the execution plan:
```
df.groupBy("country").count().explain(True)
```
This prints:
- Parsed Logical Plan
- Analyzed Logical Plan
- Optimized Logical Plan
- Physical Plan

Understanding execution plans is extremely important for performance tuning.

In [0]:
df = spark.createDataFrame([
    (1, "USA"),
    (2, "Canada"),
    (3, "USA"),
    (4, "Mexico"),
    (5, "Canada")
], ["id", "country"])

df.groupBy("country").count().explain(True)

#### Logical Plan

The Logical Plan represents the high-level description of the query.

At this stage, Spark parses the user query and converts it into a logical representation of the operations required.

Example query:
```
orders_df = spark.read.parquet(ORDERS_PATH)

result = (
    orders_df
    .filter("status = 'DELIVERED'")
    .groupBy("customer_id")
    .count()
)
```
Logical representation:
```
Scan → Filter → Aggregate
```
At this point, Spark only understands what operations need to be performed, but not how they will be executed.

In [0]:
from setup.env import *
orders_df = spark.read.parquet(ORDERS_PATH)
result = (
    orders_df
    .filter("status = 'DELIVERED'")
    .groupBy("customer_id")
    .count()
)

#### Optimization

During the optimization phase, Catalyst applies several optimization rules to improve the logical plan.

Examples of optimizations include:
- Predicate Pushdown
- Constant Folding
- Column Pruning
- Filter Reordering

Example optimization:

Original logical plan:
```
Scan → Filter → Select
```
Optimized logical plan:
```
Scan → Select → Filter
```
This reduces unnecessary data processing and improves performance.

#### Physical Plan

The Physical Plan defines how Spark will execute the query on the cluster.

At this stage Spark decides:
- Execution strategies
- Join strategies
- Partitioning strategy
- Shuffle operations

Example physical operations include:
- HashAggregate
- SortMergeJoin
- BroadcastHashJoin
- Exchange

These operations represent the actual execution steps Spark will perform.

### Join and Aggregation Strategies in Spark Physical Plans

When Spark executes a query, the Catalyst Optimizer selects the most efficient execution strategy based on:
- dataset size
- partitioning
- join conditions
- cluster configuration

These strategies appear in the Physical Plan when using:
```
df.explain(True)
```
Some of the most common execution operators include:
- HashAggregate
- SortMergeJoin
- BroadcastHashJoin

Understanding these operators is important for interpreting execution plans and optimizing Spark workloads.

#### ShuffleHashJoin - HashAggregate

HashAggregate is an aggregation strategy used by Spark to perform operations such as:
- groupBy
- count
- sum
- avg

Instead of sorting data, Spark uses a hash table in memory to group records by key.

Example aggregation:
```
orders_df.groupBy("customer_id").count()
```

Example physical plan snippet:
```
HashAggregate(keys=[customer_id], functions=[count(1)])
```

Execution process:
```
Partition data
↓
Build hash table for grouping keys
↓
Compute aggregations
```
Advantages
- Faster than sort-based aggregation
- Efficient for large datasets
- Works well when grouping keys fit in memory

Potential issue

If the aggregation requires data redistribution, Spark will perform a shuffle before the HashAggregate.

Example physical plan:
```
Exchange hashpartitioning(customer_id)
HashAggregate
```

This indicates:
```
Shuffle → Aggregation
```

#### BroadcastHashJoin

BroadcastHashJoin is a join strategy used when one dataset is small enough to be broadcast to all executors.

Instead of shuffling both datasets, Spark:
broadcasts the small table to every executor.

Example:
```
from pyspark.sql.functions import broadcast

result = orders_df.join(
    broadcast(customers_df),
    "customer_id"
)
```

Example physical plan:
```
BroadcastHashJoin
```

Execution process:
```
Small dataset broadcast to all executors
↓
Each executor performs local join
```

Advantages
- Avoids shuffle
- Very fast joins
- Ideal for dimension tables

Typical scenario:
```
fact table + small dimension table
```

Example:
```
orders JOIN customers

Where customers is small.
```

#### SortMergeJoin

SortMergeJoin is the most common join strategy used for large datasets.

Spark uses this strategy when both datasets are large and cannot be broadcast.

Execution process:
```
Shuffle both datasets
↓
Sort both datasets by join key
↓
Merge the sorted partitions
```

Example physical plan:
```
SortMergeJoin
```

Example join:
```
orders_df.join(order_items_df, "order_id")
```

Typical physical plan snippet:
```
Exchange hashpartitioning(order_id)
Sort
SortMergeJoin
```

Execution steps:
```
Shuffle
↓
Sort
↓
Merge Join
```

Advantages
- Works well for very large datasets
- Scales across distributed clusters

Disadvantages
- Requires shuffle
- Requires sorting
- More expensive than broadcast joins

#### When Spark Chooses Each Strategy (TABLE)

Spark typically selects join strategies as follows:

| Strategy          | When Used                                  |
| ----------------- | ------------------------------------------ |
| BroadcastHashJoin | One dataset is small                       |
| SortMergeJoin     | Both datasets are large                    |
| ShuffleHashJoin   | Medium datasets with compatible partitions |



Broadcast threshold is controlled by:
```
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
```
Default value is usually:
```
10 MB
```
If a table is smaller than this threshold, Spark may automatically choose BroadcastHashJoin.

### Shuffle Operations (Wide transformations)

A shuffle occurs when Spark needs to redistribute data across partitions.

Shuffle is expensive because it requires:
- Network communication
- Disk I/O
- Data serialization

Operations that trigger shuffle include:
- groupBy
- join
- distinct
- repartition
- orderBy

Example:

In [0]:
df = spark.createDataFrame([
    (1, "USA"),
    (2, "Canada"),
    (3, "USA"),
    (4, "Mexico"),
    (5, "Canada")
], ["id", "country"])

df.groupBy("country").count()

Spark must redistribute rows across partitions based on country.

## Example Demonstration

In [0]:
from setup.env import *

df = spark.read.parquet(ORDERS_PATH)

result = (
    df
    .filter("status = 'DELIVERED'")
    .groupBy("customer_id")
    .count()
)

result.show()

Execution flow:
```
filter → groupBy → count → show
```

Spark builds the DAG and executes it when show() runs.

## Exercise

Try to answer the following questions:

1️⃣ What is the difference between transformations and actions?

2️⃣ What triggers a Spark job?

3️⃣ If a DataFrame has 10 partitions, how many tasks will be created?

4️⃣ Why are shuffle operations expensive?

### Response

#### 1 - What is the difference between transformations and actions?


Transformations define operations on a DataFrame but do not execute immediately.
They return a new DataFrame and are evaluated lazily.

Examples (Narrow transformations):
- select()
- filter()
- withColumn()
- groupBy()
- join()

Actions trigger the execution of the computation and return a result to the driver or write data.

Examples (Wide transformation):
- show()
- count()
- collect()
- write()

#### 2 - What triggers a Spark Job?

A Spark Job is triggered when an Action is executed.

Example:
- df.count()

When this action runs, Spark builds the DAG and starts executing tasks.

#### 3 - If a DataFrame has 10 partitions, how many tasks will be created?

Spark will create one task per partition.

Therefore:
- 10 partitions → 10 tasks

Each task processes one partition of data.

#### 4 - Why are shuffle operations expensive?

Shuffle operations are expensive because they require:
- Data movement across the network
- Disk I/O
- Serialization and deserialization of data
- Synchronization between executors

Shuffle commonly occurs in operations such as:
- groupBy
- join
- distinct
- orderBy
- repartition

Minimizing shuffle is an important performance optimization strategy in Spark.

# 2 — Apache Spark Architecture and Components

Now that we understand the basic concepts of Spark, we can dive deeper into how Spark actually executes computations inside a cluster.

Understanding Spark architecture is essential for:
- Performance optimization
- Troubleshooting Spark jobs
- Understanding execution plans

## Spark Execution Flow

When a Spark application runs, the execution follows several steps.

High-level flow:
```
User Code
    ↓
SparkSession
    ↓
Logical Plan
    ↓
Catalyst Optimizer
    ↓
Physical Plan
    ↓
DAG Scheduler
    ↓
Task Scheduler
    ↓
Executors execute tasks
```

Each component in this pipeline plays a specific role.

## Cluster Managers

Spark itself does not manage cluster resources directly.
Instead, it relies on a Cluster Manager.

Common cluster managers:
- Standalone
- YARN
- Kubernetes
- Databricks Managed Cluster

The cluster manager is responsible for:
- Allocating resources
- Launching executors
- Managing worker nodes

## Spark Deployment Modes

Spark applications can run in different deployment modes.

### Client Mode

In Client Mode, the Driver runs on the machine submitting the job.

```
Client Machine
     |
     |---- Driver
     |
Cluster
     |
     |---- Executors
```



### Cluster Mode

In Cluster Mode, the Driver runs inside the cluster.

```
Cluster
   |
   |---- Driver
   |---- Executors
   |---- Executors
```

This is the most common mode for production workloads.

## DAG Scheduler

The DAG Scheduler is responsible for converting the logical execution plan into stages.

<b> A stage is a set of operations that can be executed without requiring shuffle. </b>

Stages are separated by shuffle boundaries.

Example pipeline:
```
read → filter → groupBy → agg
```

Execution breakdown:
```
Stage 1: read + filter
Stage 2: groupBy + aggregation
```
Because groupBy triggers a shuffle.

## Task Scheduler

Once stages are created, the Task Scheduler assigns tasks to executors.

Responsibilities:
- Distribute tasks across executors
- Handle task failures
- Retry failed tasks

Each task processes one partition of data.

## Exemple Identifying Shuffle in Execution Plans

Shuffle operations often appear in execution plans as:
```
Exchange
```

Example output snippet:
```
Exchange hashpartitioning(customer_id, 200)
```

This indicates that Spark is redistributing data across partitions.

In [0]:
from setup.env import *

df = spark.read.parquet(ORDERS_PATH)

result = (
    df
    .filter("status = 'DELIVERED'")
    .groupBy("customer_id")
    .count()
)

result.explain()

## Exercise

Question 1

Which Spark component is responsible for dividing a job into stages based on shuffle boundaries?

- A) Task Scheduler
- B) Catalyst Optimizer
- C) DAG Scheduler
- D) Tungsten Engine

Question 2

Which Spark component is responsible for assigning tasks to executors for execution?

- A) Catalyst Optimizer
- B) DAG Scheduler
- C) Task Scheduler
- D) Spark Driver

Question 3

When inspecting a Spark execution plan using:
```
df.explain(True)
```
You notice the following operation:
```
Exchange hashpartitioning(customer_id, 200)
```
What does this typically indicate?
- A) Spark is caching the DataFrame
- B) Spark is performing a shuffle operation
- C) Spark is broadcasting the dataset
- D) Spark is performing lazy evaluation


### Response

#### Question 1

Correct answer:
```
C) DAG Scheduler
```
The DAG Scheduler is responsible for:
- Converting jobs into stages
- Identifying shuffle boundaries

#### Question 2

Correct answer:
```
C) Task Scheduler
```
The Task Scheduler:
- Assigns tasks to executors
- Handles task retries
- Manages execution of tasks within each stage.

#### Question 3

Correct answer:
```
B) Spark is performing a shuffle operation
```

The Exchange operator in a physical plan usually indicates:
- Data redistribution across partitions
- Network communication between executors
- A shuffle boundary

This commonly occurs in operations like:
- groupBy
- join
- repartition
- distinct
- orderBy